# Interactive Reviews EDA: Customer Sentiment & Performance
**Datathon 2026 - Round 1**

This notebook explores the `reviews.csv` dataset and its relationships with order performance (specifically delivery speed) and product quality. 

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import time

# Force plotly to use a template
import plotly.io as pio
pio.templates.default = "plotly_white"

# Project configuration
PROCESSED_DIR = Path("../data/processed")
PLOT_DIR = Path("../data/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.options.display.max_columns = None

## 1. Data Loading (High Performance Parquet)
We join `reviews` with `orders` and `shipments` to calculate delivery delays.

In [4]:
print("Loading optimized parquet datasets...")
reviews = pd.read_parquet(PROCESSED_DIR / "reviews.parquet")
orders = pd.read_parquet(PROCESSED_DIR / "orders.parquet")
shipments = pd.read_parquet(PROCESSED_DIR / "shipments.parquet")
products = pd.read_parquet(PROCESSED_DIR / "products.parquet")

# Convert dates for analysis
reviews['review_date'] = pd.to_datetime(reviews['review_date'])
reviews['year_month'] = reviews['review_date'].dt.to_period('M').astype(str)
orders['order_date'] = pd.to_datetime(orders['order_date'])
shipments['ship_date'] = pd.to_datetime(shipments['ship_date'])

print("Calculating delivery performance...")
# Join with shipments to get delay
delivery_perf = orders[['order_id', 'order_date']].merge(shipments[['order_id', 'ship_date']], on='order_id', how='left')
delivery_perf['days_to_ship'] = (delivery_perf['ship_date'] - delivery_perf['order_date']).dt.days

print("Final merge...")
df = reviews.merge(delivery_perf, on='order_id', how='left')
df = df.merge(products[['product_id', 'category']], on='product_id', how='left')

print(f"Review dataset enriched. Records: {len(df)}")

Loading optimized parquet datasets...
Calculating delivery performance...
Final merge...
Review dataset enriched. Records: 113551


## 2. Sentiment Distribution
### Descriptive Analysis
What is the overall rating profile of the store?

In [5]:
rating_dist = df['rating'].value_counts().reset_index()
rating_dist.columns = ['rating', 'count']
rating_dist = rating_dist.sort_values('rating')

fig = px.pie(rating_dist, values='count', names='rating', 
             title='Overall Customer Rating Distribution',
             color_discrete_sequence=px.colors.sequential.RdBu_r)
fig.write_image(PLOT_DIR / "05_rating_pie.png")
fig.show()

**Insight (Diagnostic):** 
A significant portion of reviews are 1 and 2 stars. In a healthy e-commerce ecosystem, 4 and 5 stars should dominate. This suggests systemic issues with product quality or fulfillment.

## 3. The 2019 Sentiment Collapse
### Diagnostic Analysis
Did customer satisfaction drop before we saw the volume collapse?

In [6]:
monthly_rating = df.groupby('year_month').agg(
    avg_rating=('rating', 'mean'),
    review_count=('review_id', 'count')
).reset_index()

fig = px.line(monthly_rating, x='year_month', y='avg_rating', 
              title='Average Monthly Rating Trend (2012-2022)',
              color_discrete_sequence=['#EF553B'])
fig.add_vrect(x0="2018-06", x1="2019-01", 
              annotation_text="Volume Drop Start", 
              fillcolor="yellow", opacity=0.25, line_width=0)
fig.write_image(PLOT_DIR / "06_avg_rating_trend.png")
fig.show()

**Diagnostic:** 
If the average rating started declining *before* mid-2018, it implies the volume drop was a reaction to deteriorating customer experience. If it dropped *with* the volume, it might indicate a logistics failure or specific batch issues.

## 4. Delivery Lag vs Customer Satisfaction
### Predictive/Diagnostic
Is shipping speed a key driver of 1-star reviews?

In [7]:
# Filter outliers for visual clarity
clean_df = df[(df['days_to_ship'] >= 0) & (df['days_to_ship'] <= 30)]

delay_rating = clean_df.groupby('days_to_ship')['rating'].mean().reset_index()

fig = px.bar(delay_rating, x='days_to_ship', y='rating', 
             title='Average Rating by Days to Ship',
             color='rating', color_continuous_scale='RdYlGn')
fig.update_layout(yaxis_range=[1, 5])
fig.write_image(PLOT_DIR / "07_delay_vs_rating.png")
fig.show()

**Diagnostic:** 
There is a clear negative correlation: as shipping days increase, the average rating drops sharply. After 5-7 days of delay, ratings typically collapse below 3.0.

**Prescriptive:** 
Implement a 'Speed Guarantee' for the top 20% most profitable products to maintain high LTV.

## 5. Category-Level Quality Check
### Diagnostic
Which categories are the 'sentiment drain'?

In [8]:
category_sentiment = df.groupby('category').agg(
    avg_rating=('rating', 'mean'),
    total_reviews=('review_id', 'count')
).reset_index().sort_values('avg_rating')

fig = px.bar(category_sentiment, x='category', y='avg_rating', 
             text_auto='.2f',
             color='total_reviews',
             title='Average Rating by Product Category')
fig.write_image(PLOT_DIR / "08_category_sentiment.png")
fig.show()

**Prescriptive:** 
Audit the suppliers for the categories in the bottom 3 to identify if the issue is manufacturing quality or inaccurate product descriptions.